# WTGMate 장소 추출 모델 파인튜닝 (2단계)

1단계에서 Gemini로 만든 합성 데이터(`alpaca_dataset.jsonl`)로, 작은 오픈소스 모델(Qwen2.5-3B-Instruct)에
LoRA(QLoRA)를 붙여서 "자연어 일정 문장 -> 방문 장소 JSON" 태스크를 학습시킵니다.

**Colab 설정: 런타임 -> 런타임 유형 변경 -> T4 GPU (무료 티어)**

학습이 끝나면 GGUF로 내보내서 Ollama에 로드 -> 3단계에서 `backend/main.py`의 Gemini 호출을 대체합니다.

## 0. 환경 설치

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps --upgrade "git+https://github.com/unslothai/unsloth.git"

## 1. 베이스 모델 로드 (4bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # 자동 감지 (T4 -> float16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 2. LoRA 어댑터 부착

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. 데이터셋 업로드

로컬의 `backend/finetune/data/alpaca_dataset.jsonl` 을 아래 셀로 업로드하세요.
(1단계 `generate_dataset.py` 가 만든 파일 그대로 사용합니다.)

In [ ]:
from google.colab import files
uploaded = files.upload()  # alpaca_dataset.jsonl 선택
dataset_path = list(uploaded.keys())[0]
print("업로드된 파일:", dataset_path)

## 4. 프롬프트 포맷팅

학습/추론 프롬프트 형식을 통일해야 하므로, 이 노트북과 3단계(Ollama 서빙)에서
**동일한 `alpaca_prompt` 템플릿**을 그대로 재사용합니다.

In [ ]:
from datasets import load_dataset

alpaca_prompt = """다음은 작업을 설명하는 지시문과, 참고할 입력이 짝지어져 있습니다.
요청을 적절히 완료하는 응답을 작성하세요.

### 지시문:
{}

### 입력:
{}

### 응답:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

raw_dataset = load_dataset("json", data_files=dataset_path, split="train")
raw_dataset = raw_dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = raw_dataset["train"].map(formatting_prompts_func, batched=True)
eval_dataset = raw_dataset["test"].map(formatting_prompts_func, batched=True)

print(f"train: {len(train_dataset)}, eval(검증용, 학습에는 안 씀): {len(eval_dataset)}")
print(train_dataset[0]["text"])

## 5. 학습 (SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 6. 빠른 정성 평가

검증용으로 떼어둔 `eval_dataset`에서 몇 개를 직접 추론해서, JSON이 파싱되는지 /
필드가 맞는지 눈으로 확인합니다 (정식 지표는 아니고 감 잡는 용도).

In [ ]:
import json

FastLanguageModel.for_inference(model)

ok, total = 0, 0
for ex in eval_dataset.select(range(min(10, len(eval_dataset)))):
    prompt = alpaca_prompt.format(ex["instruction"], ex["input"], "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=512, use_cache=True)
    decoded = tokenizer.batch_decode(out)[0]
    generated = decoded.split("### 응답:")[-1].strip().rstrip(tokenizer.eos_token if tokenizer.eos_token else "")

    total += 1
    try:
        json.loads(generated[: generated.rfind("]") + 1])
        ok += 1
        print("[OK]", ex["input"][:40], "...")
    except Exception:
        print("[FAIL]", ex["input"][:40], "->", generated[:120])

print(f"\nJSON 파싱 성공률: {ok}/{total}")

## 7. GGUF로 내보내기 (Ollama용)

`q4_k_m` 양자화로 내보냅니다 (품질/용량 균형이 좋아 Ollama에서 표준적으로 쓰는 포맷).

In [ ]:
model.save_pretrained_gguf("wtgmate_parser", tokenizer, quantization_method="q4_k_m")

from google.colab import files as colab_files
import glob
gguf_path = glob.glob("wtgmate_parser/*.gguf")[0]
print("생성된 파일:", gguf_path)
colab_files.download(gguf_path)

## 8. (참고) 3단계 예고 - Ollama Modelfile

다운로드한 `.gguf` 파일을 로컬 PC로 가져온 뒤, 같은 폴더에 아래 내용으로 `Modelfile`을 만들고

```
FROM ./wtgmate_parser.q4_k_m.gguf
PARAMETER temperature 0.1
PARAMETER num_ctx 2048
```

다음 명령으로 로컬에 등록합니다.

```bash
ollama create wtgmate-parser -f Modelfile
ollama run wtgmate-parser
```

**주의**: 이 노트북은 챗 템플릿 없이 순수 `alpaca_prompt` 텍스트로 학습했으므로,
3단계에서 `backend/main.py`를 바꿀 때도 Ollama `/api/generate`에 채팅 템플릿이 아니라
이 노트북과 동일한 `alpaca_prompt.format(TASK_INSTRUCTION, user_text, "")` 문자열을
그대로 `prompt`로 넘겨야 합니다. 이 부분은 3단계에서 같이 진행합니다.